In [1]:
import pandas as pd
import pycountry

# Load dataset
file_path = "../data/processed/01_emdat_base_normalization.csv"
df = pd.read_csv(file_path)

# Check missing values to see if any geographic data is blank
print("Geocoding Vector Sparsity:")
display(df[['country', 'iso']].isnull().sum())

# Inspect a random sample to ensure the mappings look valid
print("\nSample of current Country-ISO mappings:")
display(df[['country', 'iso']].sample(10, random_state=42))

Geocoding Vector Sparsity:


country    0
iso        0
dtype: int64


Sample of current Country-ISO mappings:


,country,iso
3707,Mexico,MEX
2310,Panama,PAN
5284,Honduras,HND
7358,Ukraine,UKR
7831,Afghanistan,AFG
6290,Jamaica,JAM
2718,Malawi,MWI
1794,United States of America,USA
4793,China,CHN
2223,Japan,JPN


In [2]:
# Build reference ISO-3 list for validation
valid_iso_codes = [country.alpha_3 for country in pycountry.countries]

# Flag records with invalid ISO codes
invalid_records = df[~df['iso'].isin(valid_iso_codes)]

# Extract unique problematic codes for inspection
invalid_codes = invalid_records['iso'].unique()

# Report validation outcome
print(f"\nInvalid ISO codes detected: {len(invalid_codes)}")
if len(invalid_codes) > 0:
    print(f"Problematic codes: {invalid_codes}")


Invalid ISO codes detected: 3
Problematic codes: <StringArray>
['SCG', 'SPI', 'AZO']
Length: 3, dtype: str


In [3]:
# Map known EM-DAT ISO anomalies to standard codes
iso_corrections = {
    'SCG': 'SRB',  # Serbia & Montenegro → Serbia
    'SPI': 'ESP',  # Canary Islands → Spain
    'AZO': 'PRT'   # Azores → Portugal
}

# Count affected rows before fixing
affected_rows = df['iso'].isin(iso_corrections.keys()).sum()
print(f"Rows to fix: {affected_rows}")

# Apply ISO fixes
df['iso'] = df['iso'].replace(iso_corrections)

# Check if any invalid codes remain
invalid_records_after = df[~df['iso'].isin(valid_iso_codes)]
print(f"Invalid ISO codes remaining: {len(invalid_records_after)}")

Rows to fix: 18
Invalid ISO codes remaining: 0


In [4]:
import os

# Save path for processed dataset
output_path = "../data/processed/02_emdat_geocoding_standard.csv"

# Ensure output directory exists
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# Save DataFrame
df.to_csv(output_path, index=False)

# Confirm save
print(f"Saved: {output_path}")

Saved: ../data/processed/02_emdat_geocoding_standard.csv
